# Tool 5: Fine-tuning a small model on company knowledge (SFT + LoRA)

**Before running:** `Runtime` → `Change runtime type` → **T4 GPU** → Save.
Then run each cell in order (Shift+Enter).

Steps:
1. Install libraries
2. Upload `train.jsonl` and `test.jsonl`
3. Load the base model and record its answers **before** training
4. Fine-tune with LoRA
5. Record the fine-tuned model's answers on the same test questions
6. Download the results for evaluation

In [ ]:
# 1. Install libraries (takes ~1 minute)
!pip install -q -U trl peft datasets

In [ ]:
# 2. Upload train.jsonl and test.jsonl from your data/finetune folder
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 3a. Load the data
from datasets import load_dataset

data = load_dataset("json", data_files={"train": "train.jsonl", "test": "test.jsonl"})
train_ds = data["train"].remove_columns(["source"])   # keep only the "messages" column
test_ds = data["test"]
print(train_ds)
print(test_ds)
print(train_ds[0]["messages"])

In [ ]:
# 3b. Load the base model (a small open model, 0.6 billion parameters)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to("cuda")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3c. A helper that asks the model a question (no 'thinking' mode, greedy decoding = repeatable)
def ask(model, messages, max_new_tokens=150):
    prompt = [m for m in messages if m["role"] != "assistant"]   # system + question only
    inputs = tokenizer.apply_chat_template(
        prompt, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.replace("<think>", "").replace("</think>", "").strip()

# Record the BASE model's answers before any training
model.eval()
base_answers = [ask(model, row["messages"]) for row in test_ds]
for row, ans in list(zip(test_ds, base_answers))[:3]:
    print("Q:  ", row["messages"][1]["content"])
    print("REF:", row["messages"][2]["content"])
    print("BASE:", ans, "\n")

In [ ]:
# 4. Fine-tune with LoRA (trains a small add-on, not the whole model). ~3-5 minutes on a T4.
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="construction-qwen3-lora",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    max_length=512,
    assistant_only_loss=True,   # learn from the answers, not from the questions
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

In [ ]:
# 5. Record the FINE-TUNED model's answers on the same test questions
ft_model = trainer.model
ft_model.eval()
ft_answers = [ask(ft_model, row["messages"]) for row in test_ds]
for row, base, ft in list(zip(test_ds, base_answers, ft_answers))[:5]:
    print("Q:    ", row["messages"][1]["content"])
    print("REF:  ", row["messages"][2]["content"])
    print("BASE: ", base)
    print("TUNED:", ft, "\n")

In [ ]:
# 6. Save the training loss + all answers, and download them
import json

loss_history = [log for log in trainer.state.log_history if "loss" in log]
results = [
    {
        "question": row["messages"][1]["content"],
        "reference": row["messages"][2]["content"],
        "source": row["source"],
        "base_answer": base,
        "finetuned_answer": ft,
    }
    for row, base, ft in zip(test_ds, base_answers, ft_answers)
]
with open("finetune_results.json", "w") as f:
    json.dump({"model": MODEL_ID, "loss_history": loss_history, "results": results}, f, indent=2)

trainer.model.save_pretrained("construction-qwen3-lora")   # the LoRA adapter (only a few MB)
!zip -qr construction-qwen3-lora.zip construction-qwen3-lora
files.download("finetune_results.json")
files.download("construction-qwen3-lora.zip")